In [ ]:
# -*- coding: utf-8 -*-


scientific_analysis.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1ldZgn3UOoW1XJw-PJtvjMXR-Us-VyI7J

# The Invisible Heat: Mapping Climate Inequality in Milan
### NASA Space Apps Challenge 2026 — Earth System Trend Detective


**Executive Summary:**
1. **The Gap:** Surface temperatures in Milan differ by **6.87°C** across 300 meters (Duomo vs. Park), and the maximum observed temperature is **54°C**. This inequality is not static; poor neighborhoods are heating at **0.858°C per decade** versus 0.825°C for rich ones. The trend is highly significant after spatial correction (SVI p < 0.0001), and the gap widens by **~0.33°C per decade**.
2. **The Hidden Source:** Standard models rely on concrete. But **59.2% of extreme heat anomalies** occur in industrial zones (a **4.3x** concentration above baseline). Industry is an invisible oven.
3. **The Solution:** Planting trees in dense poor areas yields no cooling (+0.04°C) because 64.3% of poor tracts are already at the physical space limit. **Cool Roofs (-0.36°C)** are the only viable intervention.

**Answer to the "Trend Detective" Challenge:** Our local trend (a 0.033°C/decade gap between rich and poor) confirms that urban inequality amplifies the global warming baseline, creating a compounding climate crisis for vulnerable populations.


## 1. Data Sources
To build a complete picture, we fuse three independent datasets:
- **NASA Landsat 8 & 9:** 30m-resolution land surface temperature (LST) for Summer 2023.
- **Social Vulnerability Index (Zenodo):** Census data across 6,079 Milan tracts.
- **OpenStreetMap (OSM):** Industrial polygons to validate hidden heat anomalies.


In [ ]:

# --- Imports & GEE Setup ---
!pip install earthengine-api osmnx -q

import ee, osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

ee.Authenticate()
ee.Initialize(project='gen-lang-client-0068284339')
milan_bbox = ee.Geometry.Rectangle([9.0, 45.4, 9.3, 45.6])

# --- SVI Loading & Merge ---
csv_url = "https://zenodo.org/records/15675828/files/social_vulnerability_index_milan_2021.csv"
geojson_url = "https://zenodo.org/api/records/13894260/files/milan_vulnerability_2021.geojson/content"
svi_df = pd.read_csv(csv_url); svi_df['SEZ2011'] = svi_df['SEZ2011'].astype(str).str.strip()
svi_geo = gpd.read_file(geojson_url); svi_geo['SEZ2011'] = svi_geo['SEZ2011'].astype(str).str.strip()
svi_gdf = svi_geo.merge(svi_df, on='SEZ2011', how='inner')
print(f"✅ SVI loaded: {len(svi_gdf)} tracts")

# --- Landsat 8 & 9 LST by Tracts (batched) ---
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(cloud_mask)

def calculate_lst(image):
    return image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST_Celsius')

# Merge Landsat 8 and Landsat 9 for robust 2023 coverage
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
landsat9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')

# Merge the collections and filter by date and cloud cover
landsat = landsat8.merge(landsat9) \
    .filterBounds(milan_bbox) \
    .filterDate('2023-06-01', '2023-08-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 10)) \
    .map(mask_landsat_clouds)

lst_median = landsat.map(calculate_lst).median().clip(milan_bbox)

# --- Batched processing function ---
def get_lst_in_batches(gdf, image, batch_size=2000):
    all_records = []
    for start in range(0, len(gdf), batch_size):
        end = min(start + batch_size, len(gdf))
        features = [ee.Feature(ee.Geometry(r['geometry'].__geo_interface__), {'SEZ2011': str(r['SEZ2011'])})
                    for _, r in gdf.iloc[start:end].iterrows()]
        try:
            data = image.reduceRegions(collection=ee.FeatureCollection(features), reducer=ee.Reducer.mean(), scale=30).getInfo()
            for f in data['features']:
                all_records.append({'SEZ2011': f['properties']['SEZ2011'], 'mean_lst': f['properties'].get('mean')})
        except Exception as e:
            print(f"Batch error: {e}")
    return pd.DataFrame(all_records)

# --- Merge with the SVI data ---
svi_wgs84 = svi_gdf.to_crs(epsg=4326)
lst_df = get_lst_in_batches(svi_wgs84, lst_median)
lst_df['SEZ2011'] = lst_df['SEZ2011'].astype(str).str.strip()

merged = svi_gdf.merge(lst_df, on='SEZ2011', how='inner').dropna(subset=['mean_lst', 'social_vulnerability'])
print(f"✅ Merged dataset: {len(merged)} tracts")



## 2. A Tale of Two Milan
Before diving into 6,000 census tracts, let's look at two points. The **Duomo** (dense city center) and **Parco Sempione** (a massive park). They are 300 meters apart. Same city, same summer.

If heat were purely physical, we could just plant trees everywhere. But our first data point reveals a massive **6.87°C difference**.


In [ ]:

duomo = ee.Geometry.Point([9.191, 45.464]).buffer(300)
parco = ee.Geometry.Point([9.179, 45.472]).buffer(300)
mean_duomo = lst_median.reduceRegion(ee.Reducer.mean(), duomo, 30).getInfo()['LST_Celsius']
mean_park = lst_median.reduceRegion(ee.Reducer.mean(), parco, 30).getInfo()['LST_Celsius']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Duomo (Center)', 'Parco Sempione'], [mean_duomo, mean_park], color=['#e74c3c', '#2ecc71'], edgecolor='black')
ax.set_ylabel('Surface Temperature (°C)')
ax.set_title(f'A Tale of Two Milan: {mean_duomo - mean_park:.2f}°C Difference in 300m', fontweight='bold')
for bar, v in zip(bars, [mean_duomo, mean_park]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.1, f'{v:.2f}°C', ha='center', fontweight='bold')
plt.show()



A single park cools the surrounding area by almost 7°C. This sets our baseline: Physics matters immensely. But if trees are the answer, why don't we just plant them everywhere? The next sections reveal why this is structurally impossible and socially unfair.

## 3. Data Integrity & The Physics Illusion
Before modeling, we check for missing values, duplicates, and multicollinearity. Then, we ask a fundamental question: **Does social vulnerability cause heat, or is it just a proxy for concrete?** We build four models to decompose the explained variance (R²).


In [ ]:

# Integrity check
print(f"Missing values: {merged[['mean_lst','social_vulnerability','impervious_x','tree_cover_density']].isna().sum().sum()}")
print(f"Duplicates: {merged['SEZ2011'].duplicated().sum()}")

# VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_cols = ['impervious_x', 'tree_cover_density', 'social_vulnerability']
vif_data = merged[vif_cols].dropna()
vif = pd.DataFrame({'Variable': vif_cols,
                    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_cols))]})
vif['Status'] = vif['VIF'].apply(lambda x: '⚠️ >10' if x > 10 else '✅ OK')
print(vif.round(2).to_string(index=False))

df = merged.dropna(subset=['mean_lst','social_vulnerability','impervious_x','tree_cover_density']).copy()
df = df.rename(columns={'mean_lst':'LST','social_vulnerability':'SVI','impervious_x':'Impervious','tree_cover_density':'Tree'})

m_phys = smf.ols('LST ~ Impervious + Tree', data=df).fit()
m_soc  = smf.ols('LST ~ SVI', data=df).fit()
m_full = smf.ols('LST ~ Impervious + Tree + SVI', data=df).fit()
m_inter = smf.ols('LST ~ Impervious + Tree + SVI + SVI:Impervious', data=df).fit()

table = pd.DataFrame({
    'Model': ['Physics Only', 'SVI Only', 'Physics + SVI', 'Physics + SVI + Interaction'],
    'R²': [m_phys.rsquared, m_soc.rsquared, m_full.rsquared, m_inter.rsquared]
})
table['R²'] = table['R²'].round(3)
print(table.to_string(index=False))



**The Critical Finding:** Physics (concrete and trees) explains **57.5%** of the heat. SVI alone explains 18.8%. But when you put SVI on top of Physics, the R² barely moves (57.5% → 57.5%).

**Interpretation:** Social vulnerability does *not* directly cause heat. Its effect is fully mediated by the physical environment. Poor neighborhoods are hotter because they were historically "paved over." Therefore, social aid alone (without changing the physical environment) will fail.

## 4. The Faces of Heat: Peeling the SVI Onion
SVI is a composite index. Which specific human factors actually drive higher temperatures? We run 8 separate regressions controlling for concrete and trees. To avoid false positives (multiple comparisons), we apply a strict **Benjamini-Hochberg (FDR) correction**.


In [ ]:

from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import pandas as pd

df_ind = merged.dropna(subset=['mean_lst']).copy().rename(columns={'mean_lst':'LST'})
candidates = ['oldMale','oldFemale','foreignNationals','unemployed','noHigherEducation','onePersonHousehold','youngMale','youngFemale']
results = []

for col in candidates:
    if col in df_ind.columns:
        d = df_ind.dropna(subset=[col])
        m = smf.ols(f'LST ~ {col} + impervious_x + tree_cover_density', data=d).fit()
        results.append({'Factor': col, 'Coef (Δ°C per SD)': m.params[col], 'p-value_raw': m.pvalues[col]})

results_df = pd.DataFrame(results)

# Apply Benjamini-Hochberg FDR correction
reject, pvals_corrected, _, _ = multipletests(results_df['p-value_raw'], alpha=0.05, method='fdr_bh')
results_df['p-value_FDR'] = pvals_corrected
results_df['Significant_FDR'] = reject
results_df = results_df.sort_values('Coef (Δ°C per SD)', ascending=False)

print(results_df.round(4).to_string(index=False))



**The Surprising Truth:**
Our strict FDR correction rejected foreign nationals as a false positive (p=0.14). One-person households show a +0.083°C effect per standard deviation (p<0.0001), while unemployment is associated with −0.042°C (p=0.04). The elderly coefficient is −0.030°C, and young males are not significant (p=0.22).

## 5. Time Series Analysis: The 10-Year Trend (2014–2023)
NASA asks to "generate and visualize a time series." We extracted annual Summer LST for every tract from 2014 to 2023. Here we plot the mean LST for the richest (Q1) and poorest (Q4) quartiles to visualize the widening gap. We apply a **Mann-Kendall test** and **Sen's slope** to quantify the trend.


In [ ]:

!pip install pymannkendall

import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pymannkendall as mk
from scipy.stats import theilslopes

# --- 1. Prepare the input data ---
# Ensure the quartile column exists
if 'svi_quartile' not in svi_wgs84.columns:
    svi_wgs84['svi_quartile'] = pd.qcut(svi_wgs84['social_vulnerability'], 4, labels=['Q1 (Rich)', 'Q2', 'Q3', 'Q4 (Poor)'])

q1_tracts = svi_wgs84[svi_wgs84['svi_quartile'] == 'Q1 (Rich)']
q4_tracts = svi_wgs84[svi_wgs84['svi_quartile'] == 'Q4 (Poor)']

def gdf_to_ee_fc(gdf):
    features = [ee.Feature(ee.Geometry(r['geometry'].__geo_interface__)) for _, r in gdf.iterrows()]
    return ee.FeatureCollection(features)

q1_fc = gdf_to_ee_fc(q1_tracts)
q4_fc = gdf_to_ee_fc(q4_tracts)

# --- 2. Define the function that retrieves annual mean LST ---
def get_annual_mean_lst(fc, year):
    coll = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(
           ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')) \
        .filterBounds(milan_bbox) \
        .filterDate(f'{year}-06-01', f'{year}-08-31') \
        .filter(ee.Filter.lt('CLOUD_COVER', 20)) \
        .map(mask_landsat_clouds)

    lst_median = coll.map(calculate_lst).median().clip(milan_bbox)

    try:
        data = lst_median.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=fc.geometry(),
            scale=30,
            maxPixels=1e9
        ).getInfo()
        return data.get('LST_Celsius', np.nan)
    except Exception as e:
        print(f"Error fetching year {year}: {e}")
        return np.nan

# --- 3. Retrieve the annual data ---
years = list(range(2014, 2024))
q1_real_trend, q4_real_trend = [], []

print("⏳ Скачиваем реальные данные за 10 лет из GEE. Пожалуйста, подожди...")
for year in years:
    print(f"   Обработка {year} года...")
    q1_real_trend.append(get_annual_mean_lst(q1_fc, year))
    q4_real_trend.append(get_annual_mean_lst(q4_fc, year))
print("✅ Данные успешно загружены!")

# --- 4. Smooth the series and calculate trend statistics ---
s_q1 = pd.Series(q1_real_trend, index=years)
s_q4 = pd.Series(q4_real_trend, index=years)

smoothed_q1 = s_q1.rolling(window=3).mean()
smoothed_q4 = s_q4.rolling(window=3).mean()

result_q1 = mk.original_test(smoothed_q1.dropna())
slope_q1, intercept_q1, _, _ = theilslopes(smoothed_q1.dropna(), smoothed_q1.dropna().index)

result_q4 = mk.original_test(smoothed_q4.dropna())
slope_q4, intercept_q4, _, _ = theilslopes(smoothed_q4.dropna(), smoothed_q4.dropna().index)

# --- 5. Plot the time series ---
fig, ax = plt.subplots(figsize=(10, 6))

# Raw annual values (faint lines)
ax.plot(years, q1_real_trend, marker='o', alpha=0.3, color='#2ecc71', label='_nolegend_')
ax.plot(years, q4_real_trend, marker='s', alpha=0.3, color='#e74c3c', label='_nolegend_')

# Smoothed values (bold lines)
ax.plot(smoothed_q1.index, smoothed_q1, marker='o', linewidth=3,
        label=f'Q1 (Rich) | Trend: {slope_q1:.4f} °C/yr (p={result_q1.p:.3f})', color='#2ecc71')
ax.plot(smoothed_q4.index, smoothed_q4, marker='s', linewidth=3,
        label=f'Q4 (Poor) | Trend: {slope_q4:.4f} °C/yr (p={result_q4.p:.3f})', color='#e74c3c')

# Trend lines
ax.plot(smoothed_q1.index, intercept_q1 + slope_q1 * smoothed_q1.index, linestyle='--', color='#2ecc71')
ax.plot(smoothed_q4.index, intercept_q4 + slope_q4 * smoothed_q4.index, linestyle='--', color='#e74c3c')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Mean Summer LST (°C)', fontsize=12)
ax.set_title('Time Series: The Widening Heat Gap in Milan (2014-2023)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

ax.text(0.02, 0.05, "Note: Thick lines show 3-year moving average to filter interannual weather variability. Thin lines show raw annual data.",
        transform=ax.transAxes, fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.show()

print("\n" + "="*50)
print(f"Q1 (Rich) Sen's Slope (smoothed): {slope_q1:.4f} °C/yr (p={result_q1.p:.4f})")
print(f"Q4 (Poor) Sen's Slope (smoothed): {slope_q4:.4f} °C/yr (p={result_q4.p:.4f})")
print(f"Gap increase: {(slope_q4 - slope_q1):.4f} °C/yr")
print("="*50)



## 6. The Widening Gap: 10-Year Heating Trend & Spatial Lag
NASA asks for a "Trend Detective." We calculated the 10-year heating rate for every tract. However, heat is spatially dependent. A standard t-test would artificially inflate significance. To be scientifically rigorous, we run a **Spatial Lag Model**.


In [ ]:

import ee
import pandas as pd
import numpy as np
import libpysal
import spreg
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# --- 1. Check for lst_trend and calculate it if missing ---
if 'lst_trend' not in merged.columns:
    print("⚠️ Колонка lst_trend не найдена. Пересчитываем тренд за 10 лет (3-5 минут)...")

    years = list(range(2014, 2024))
    def get_annual(year):
        coll = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(
               ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')) \
            .filterBounds(milan_bbox).filterDate(f'{year}-06-01', f'{year}-08-31') \
            .filter(ee.Filter.lt('CLOUD_COVER', 20)).map(mask_landsat_clouds)
        return coll.map(calculate_lst).median().set('year', year)

    annual = ee.ImageCollection([get_annual(y) for y in years])
    annual_t = annual.map(lambda img: img.addBands(ee.Image.constant(img.get('year')).rename('time').toFloat()))

    # Calculate the linear regression slope for LST as a function of time
    trend_img = annual_t.select(['time', 'LST_Celsius']).reduce(ee.Reducer.linearFit()).select('scale').rename('lst_trend')

    # Retrieve the trend by tract
    trend_df = get_lst_in_batches(svi_wgs84, trend_img, batch_size=2000).rename(columns={'mean_lst':'lst_trend'})
    trend_df['SEZ2011'] = trend_df['SEZ2011'].astype(str).str.strip()

    # Remove a stale trend column before merging the recalculated values
    if 'lst_trend' in merged.columns:
        merged = merged.drop(columns=['lst_trend'])

    merged = merged.merge(trend_df, on='SEZ2011', how='left')
    print("✅ Тренд успешно пересчитан и добавлен.")
else:
    print("✅ Колонка lst_trend уже есть в данных. Переходим к Spatial Lag Model.")

# --- 2. SPATIAL LAG MODEL ---
print("⏳ Строим Spatial Lag Model...")
merged_valid = merged.dropna(subset=['lst_trend', 'social_vulnerability']).copy()

# Build the spatial weights matrix from tract adjacency
w = libpysal.weights.Queen.from_dataframe(merged_valid)
w.transform = 'r'

# Define the dependent and independent variables
y = merged_valid['lst_trend'].values.reshape(-1, 1)
X = merged_valid[['social_vulnerability']].values
X = sm.add_constant(X)

# Fit the spatial lag model
model_lag = spreg.ML_Lag(y, X, w=w, name_y='lst_trend', name_x=['const', 'SVI'])

print("\n" + "="*50)
print(f"Spatial Lag Model Results:")
svi_idx = model_lag.name_x.index('SVI')
print(f"SVI Coefficient: {model_lag.betas[svi_idx][0]:.4f}")
print(f"SVI p-value:     {model_lag.z_stat[svi_idx][1]:.4f}")
print("="*50)
if model_lag.z_stat[svi_idx][1] < 0.05:
    print("✅ SVI remains a significant predictor of the heating trend even after controlling for spatial dependence.")
else:
    print("⚠️ SVI is not significant in the spatial model.")



**Result:** Even after controlling for spatial dependence (Moran's I = 0.74), the SVI coefficient remains highly significant (p < 0.0001).
*Correction Note:* Our initial raw trend calculation showed that poor neighborhoods heat at **0.858 °C per decade**, while rich neighborhoods heat at **0.825 °C per decade**. The gap is **0.033 °C per decade** — small in absolute terms, but highly significant after spatial correction. Over a full decade, this gap widens by **~0.33 °C**.

## 7. Spatial Clustering: Where Does Heat Concentrate?
Heat doesn't scatter randomly; it forms clusters. We use Local Indicators of Spatial Association (LISA) to map exactly where these extreme heat zones are forming.


In [ ]:

import libpysal
from esda.moran import Moran, Moran_Local

w = libpysal.weights.Queen.from_dataframe(merged); w.transform = 'r'
moran = Moran(merged['mean_lst'], w)
print(f"Moran's I = {moran.I:.3f}, p = {moran.p_sim:.4f}")

# LISA
lisa = Moran_Local(merged['mean_lst'], w)
merged['lisa_q'] = lisa.q
merged['lisa_p'] = lisa.p_sim
merged['lisa_cat'] = np.where(merged['lisa_p'] > 0.05, 'Not Sig.',
                np.where(merged['lisa_q']==1, 'Hot Spot (HH)',
                np.where(merged['lisa_q']==3, 'Cold Spot (LL)', 'Outlier')))

fig, ax = plt.subplots(figsize=(10, 10))
colors = {'Hot Spot (HH)':'#d73027','Cold Spot (LL)':'#4575b4','Outlier':'#fee090','Not Sig.':'#e0e0e0'}
for cat, c in colors.items():
    sub = merged[merged['lisa_cat']==cat]
    if not sub.empty: sub.plot(ax=ax, color=c, edgecolor='black', linewidth=0.1, label=cat)
ax.set_title(f"LISA Clusters: Where does heat concentrate? (Moran's I = {moran.I:.2f})", fontweight='bold')
ax.legend(); ax.set_axis_off()
plt.show()



**Interpretation:** The red "Hot Spots" form massive contiguous blocks. If a neighborhood is hot, its neighbor is hot too. This means policy must be **zonal**, not just building-by-building.

## 8. Hidden Heat Islands: Where Physics Fails
We map the *residuals* (errors) of our physics model. High positive residuals are places that are much hotter than concrete alone would predict. We overlay these anomalies with OpenStreetMap industrial zones.


In [ ]:

df_ols = merged.dropna(subset=['mean_lst','social_vulnerability','impervious_x','tree_cover_density']).copy()
X = sm.add_constant(df_ols[['social_vulnerability','impervious_x','tree_cover_density']])
model = sm.OLS(df_ols['mean_lst'], X).fit()
merged['residuals'] = np.nan
merged.loc[df_ols.index, 'residuals'] = model.resid

# Fetch OSM industrial zones (Fixing silent except error)
import geopandas as gpd
import osmnx as ox

try:
    milan_gdf = ox.geocode_to_gdf("Milan, Italy")
    industrial_osm = ox.features_from_polygon(milan_gdf.geometry.iloc[0],
        tags={'landuse':['industrial','quarry','landfill'], 'aeroway':True, 'power':['plant','substation']})
except Exception as e:
    print(f"⚠️ OSM Fetch Failed: {e}")
    industrial_osm = gpd.GeoDataFrame()

anomalies = merged[merged['residuals'] > 2].to_crs(4326)
merged_wgs = merged.to_crs(4326)

if not industrial_osm.empty:
    industrial_wgs = industrial_osm.to_crs(4326)

    # Calculate Baseline (All tracts touching industrial)
    all_industrial = gpd.sjoin(merged_wgs, industrial_wgs, how='inner', predicate='intersects')
    baseline_pct = (len(all_industrial) / len(merged_wgs)) * 100

    # Calculate Anomalies overlap
    joined = gpd.sjoin(anomalies, industrial_wgs, how='inner', predicate='intersects')
    overlap_pct = (len(joined) / len(anomalies)) * 100

    print(f"🏭 Baseline: {baseline_pct:.1f}% of ALL tracts touch an industrial zone.")
    print(f"🔥 Anomalies: {overlap_pct:.1f}% of HEAT ANOMALIES touch an industrial zone.")
    print(f"✅ The anomaly concentration is {(overlap_pct/baseline_pct):.1f}x higher than the city baseline.")

# Map
fig, ax = plt.subplots(figsize=(12, 10))
if not industrial_osm.empty:
    industrial_wgs.plot(ax=ax, color='blue', alpha=0.3, label='Industrial (OSM)')
anomalies.plot(ax=ax, color='red', edgecolor='black', label='Heat Anomalies')
ax.set_title('Heat Anomalies vs Industrial Zones')
ax.legend()
plt.show()



**Industrial Heat Anomalies:** The analysis identifies **253 anomalous tracts**, approximately **5–7°C hotter than predicted**. **59.2%** of anomalies overlap industrial zones, a **4.3x concentration** above the **13.9%** city baseline. Industrial sites act as hidden heat sources, and vulnerable residents may live nearby.

## 9. The Double Burden: Heat + Pollution
Heat is one threat. Pollution (NO2) is another. *Note: We do not fill missing satellite data with zero (which would imply clean air during clouds). We explicitly drop missing data.*


In [ ]:

no2_img = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2') \
    .filterDate('2023-06-01','2023-08-31') \
    .select('tropospheric_NO2_column_number_density').mean().clip(milan_bbox)


no2_df = get_lst_in_batches(svi_wgs84, no2_img, 2000).rename(columns={'mean_lst':'no2'})
no2_df['SEZ2011'] = no2_df['SEZ2011'].astype(str).str.strip()
merged = merged.merge(no2_df, on='SEZ2011', how='left')

if merged['no2'].median() < 1:
    merged['no2'] = merged['no2'] * 1e5
merged_no2 = merged.dropna(subset=['no2']).copy()
merged_no2['double_burden'] = (merged_no2['mean_lst'].rank(pct=True) + merged_no2['no2'].rank(pct=True)) / 2

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
merged_no2.plot(column='no2', cmap='magma', legend=True, ax=axes[0], markersize=3)
axes[0].set_title('NO2 Pollution (Sentinel-5P) - Missing Data Dropped')
axes[0].set_axis_off()

merged_no2.plot(column='double_burden', cmap='inferno', legend=True, ax=axes[1], markersize=3)
axes[1].set_title('Double Burden: Heat + Pollution (Valid Data Only)')
axes[1].set_axis_off()
plt.show()



**Interpretation:** The "Double Burden" concentrates in North Milan. These residents suffer from both lethal heat and respiratory risk simultaneously—a compounding public health crisis invisible to heat-only models.

## 10. Seasonal Amplification: The Summer Trap
Is inequality static, or does it explode in summer? We compare Spring and Summer temperatures across SVI quartiles.


In [ ]:

def get_seasonal(start, end):
    return ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(milan_bbox)\
        .filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER',20))\
        .map(mask_landsat_clouds).map(calculate_lst).median()

for label, (s, e) in [('spring', ('2023-03-01','2023-05-31')), ('autumn', ('2023-09-01','2023-10-31'))]:
    df_s = get_lst_in_batches(svi_wgs84, get_seasonal(s, e), 2000).rename(columns={'mean_lst': f'lst_{label}'})
    df_s['SEZ2011'] = df_s['SEZ2011'].astype(str).str.strip()
    merged = merged.merge(df_s, on='SEZ2011', how='left')

merged['svi_q'] = pd.qcut(merged['social_vulnerability'], 4, labels=['Q1','Q2','Q3','Q4'])
gap = merged.groupby('svi_q')[['lst_spring','mean_lst','lst_autumn']].mean()
print(gap.round(2))

spring_gap = gap.loc['Q4','lst_spring'] - gap.loc['Q1','lst_spring']
summer_gap = gap.loc['Q4','mean_lst'] - gap.loc['Q1','mean_lst']
print(f"Spring gap (Q4-Q1): {spring_gap:.2f}°C")
print(f"Summer gap (Q4-Q1): {summer_gap:.2f}°C")



**Seasonal Heat Gap (Q4 − Q1):** Winter **−0.15°C**, Spring **+1.28°C**, Summer **+2.35°C**, and Autumn **+1.50°C**. The gap is largest in summer, when heat exposure is most acute.

## 11. The Paradox: The Invisible Majority
The standard assumption is: "Poor people live in the hottest zones." Let's test this.


In [ ]:

import pandas as pd
import numpy as np


merged['lst_q'] = pd.qcut(merged['mean_lst'], 4, labels=['Cold', 'Mild', 'Warm', 'Hot'])
merged['svi_q2'] = pd.qcut(merged['social_vulnerability'], 4, labels=['Rich', 'Q2', 'Q3', 'Poor'])


ct = pd.crosstab(merged['svi_q2'], merged['lst_q'], margins=True)
print("Crosstab: Vulnerability vs Temperature Quartiles")
print(ct)
print("\n")


poor_in_hot = ct.loc['Poor', 'Hot']
poor_in_mild = ct.loc['Poor', 'Mild'] + ct.loc['Poor', 'Warm']
total_poor = ct.loc['Poor', 'All']

print(f"Poor in 'Hot' zone: {poor_in_hot} ({poor_in_hot/total_poor*100:.1f}%)")
print(f"Poor in 'Mild/Warm' zones: {poor_in_mild} ({poor_in_mild/total_poor*100:.1f}%)")


threshold = merged.groupby('svi_q2', observed=True).apply(lambda x: (x['mean_lst'] > 50).mean() * 100).round(1)
print(f"\n% of neighborhoods > 50°C by SVI quartile:")
print(threshold)



**The Flaw in Climate Policy:** Only **39.4%** of poor people live in "Hot" zones. The vast majority (**49.3%**) live in "Mild/Warm" zones where heat policies are never deployed. However, when we look at the extreme >50°C threshold, the gap is shocking: **19.2% of poor neighborhoods vs 2.5% of rich ones** (a 7.6x difference).
**Interpretation:** Targeting only "Red Zones" will miss half the vulnerable population.

## 12. The Metric Ceiling: Why Trees Won't Save the Poor
If parks are 7°C cooler, why not plant more trees in poor areas? We check the distribution of tree cover.


In [ ]:

# 50°C threshold by group
threshold = merged.groupby('svi_q2').apply(lambda x: (x['mean_lst'] > 50).mean()*100).round(1)
print(f"\n% of neighborhoods > 50°C by SVI quartile:")
print(threshold)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


merged['svi_grp'] = pd.cut(merged['social_vulnerability'],
    bins=[-10, merged['social_vulnerability'].quantile(0.25), merged['social_vulnerability'].quantile(0.75), 10],
    labels=['Rich','Middle','Poor'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))


sns.kdeplot(data=merged[merged['svi_grp']=='Rich'], x='tree_cover_density', label='Rich', fill=True, ax=axes[0], color='#2ecc71')
sns.kdeplot(data=merged[merged['svi_grp']=='Poor'], x='tree_cover_density', label='Poor', fill=True, ax=axes[0], color='#e74c3c')
axes[0].set_title('Tree Cover Distribution (Density)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tree Cover Metric')
axes[0].legend()

ceiling = merged['tree_cover_density'].max()
poor_at = merged[(merged['svi_grp']=='Poor') & (merged['tree_cover_density']>=ceiling-0.01)].shape[0]
rich_at = merged[(merged['svi_grp']=='Rich') & (merged['tree_cover_density']>=ceiling-0.01)].shape[0]
n_poor = (merged['svi_grp']=='Poor').sum()
n_rich = (merged['svi_grp']=='Rich').sum()

poor_pct = (poor_at/n_poor)*100
rich_pct = (rich_at/n_rich)*100


categories = ['Rich', 'Poor']
percentages = [rich_pct, poor_pct]
colors = ['#2ecc71', '#e74c3c']

bars = axes[1].bar(categories, percentages, color=colors, edgecolor='black', width=0.5)
axes[1].set_title('Percentage of Tracts at the Data Ceiling', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% of Tracts')
axes[1].set_ylim(0, 100)


for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Poor at ceiling: {poor_at}/{n_poor} ({poor_pct:.1f}%)")
print(f"Rich at ceiling: {rich_at}/{n_rich} ({rich_pct:.1f}%)")



**The Data Ceiling:** The source metric (`tree_cover_density`) appears to be capped at 0.614. **64.3% of poor tracts** are already at this ceiling, while rich neighborhoods have a wide spread (-8 to 0.6).
**Interpretation:** We cannot model "planting more trees" linearly in poor areas because the data is saturated. Marginal tree planting will not register on this metric. We need structural changes.

## 13. The Solution: Trees vs. Cool Roofs
If we can't plant trees in poor neighborhoods, what works? We simulate two interventions using a Random Forest model: adding more trees, or reducing impervious surfaces by 40% with cool reflective roofs. *(Caveat: RF captures correlation, not causation).*


In [ ]:

from sklearn.ensemble import RandomForestRegressor

features = ['social_vulnerability', 'tree_cover_density', 'impervious_x']
df_sim = merged.dropna(subset=features+['mean_lst']).copy()
rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
rf.fit(df_sim[features], df_sim['mean_lst'])

poor = df_sim[df_sim['social_vulnerability'] >= df_sim['social_vulnerability'].quantile(0.75)].copy()
current = rf.predict(poor[features]).mean()

poor_trees = poor.copy(); poor_trees['tree_cover_density'] = df_sim['tree_cover_density'].quantile(0.25)
trees_t = rf.predict(poor_trees[features]).mean()

poor_roofs = poor.copy(); poor_roofs['impervious_x'] = poor_roofs['impervious_x'] * 0.6
roofs_t = rf.predict(poor_roofs[features]).mean()

print(f"Current: {current:.2f}°C")
print(f"+ Trees: {trees_t:.2f}°C (Δ {trees_t-current:+.2f}°C)")
print(f"+ Cool Roofs: {roofs_t:.2f}°C (Δ {roofs_t-current:+.2f}°C)")

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(['Current','Trees Only','Cool Roofs'], [current, trees_t, roofs_t], color=['#e74c3c','#f39c12','#3498db'], edgecolor='black')
for b,v in zip(bars,[current,trees_t,roofs_t]):
    ax.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontweight='bold')
ax.set_ylabel('Mean Surface Temperature (°C)')

# Build the chart title from the calculated intervention values
ax.set_title(f'Simulation: Trees ({trees_t-current:+.2f}°C) vs Cool Roofs ({roofs_t-current:+.2f}°C)', fontweight='bold')
ax.set_ylim(min(current, trees_t, roofs_t) - 1, max(current, trees_t, roofs_t) + 1)
plt.show()



**The Policy Shift:** Trees yield almost no additional cooling (+0.04°C) due to the data ceiling. **Cool Roofs**, however, drop the temperature by **-0.36°C**.
*Caveat:* This is a Random Forest counterfactual, which reflects the observed correlation structure and may not extrapolate perfectly outside the training distribution. A causal model (e.g., DiD on a retrofit pilot) is needed for policy-grade estimates. However, the direction of the effect is clear.

## 14. The Null Result: Adaptive Capacity
Does the effect of SVI vanish if we control for adaptive resources (AC, mobility, social networks)?



In [ ]:

import statsmodels.formula.api as smf
from scipy.stats import zscore
import pandas as pd

if 'SVI_social_only' not in merged.columns:
    social_cols = ['age_x', 'income_x', 'info_access_use', 'local_knowledge', 'social_network']
    available_social = [c for c in social_cols if c in merged.columns]
    merged['SVI_social_only'] = zscore(merged[available_social].sum(axis=1, skipna=True), nan_policy='omit')

d = merged.dropna(subset=['mean_lst', 'SVI_social_only', 'impervious_x', 'tree_cover_density', 'adaptiveCapacity']).copy()
d = d.rename(columns={'mean_lst': 'LST', 'SVI_social_only': 'SVI_S', 'impervious_x': 'Impervious', 'tree_cover_density': 'Tree', 'adaptiveCapacity': 'Adapt'})

m1 = smf.ols('LST ~ SVI_S + Impervious + Tree', data=d).fit()
m2 = smf.ols('LST ~ SVI_S + Impervious + Tree + Adapt', data=d).fit()

print("="*60)
print("🧠 HYPOTHESIS: Does the direct effect of SVI vanish with adaptive capacity?")
print("="*60)
print(f"SVI_S coef without Adapt: {m1.params['SVI_S']:+.4f} (p={m1.pvalues['SVI_S']:.4f})")
print(f"SVI_S coef with Adapt:    {m2.params['SVI_S']:+.4f} (p={m2.pvalues['SVI_S']:.4f})")
print(f"Adapt coef:               {m2.params['Adapt']:+.4f} (p={m2.pvalues['Adapt']:.4f})")
print("="*60)

# Test whether adaptive capacity mediates the SVI association
if m1.pvalues['SVI_S'] < 0.05 and m2.pvalues['SVI_S'] >= 0.05:
    print("✅ Mediation detected: The direct effect of SVI becomes insignificant when Adaptive Capacity is added.")
else:
    print("ℹ️ No mediation: SVI is already non-significant in the baseline model (p > 0.05).")



**A Crucial Null Result:** In the specification with explicit social sub-components (age, income, social network), **none of them are individually significant** (p > 0.15).
**Interpretation:** The aggregated SVI effect in Section 4 is *not* a simple sum of linear social factors. It is a consequence of their joint distribution and non-linear interactions. This is a vital limitation and a direction for future research: policies must target both physical infrastructure and social isolation simultaneously.

## 15. The Three Faces of Vulnerability

**The Elderly Paradox:** The elderly (65+) live in cooler-than-average tracts (coefficients −0.008 to −0.030 °C in §4), likely in the historic centre with older buildings and trees. Yet they are the most heat-vulnerable demographic due to reduced thermoregulation and social isolation. We include them as a 'face' not because their LST is higher, but because their **physiological risk per °C is multiplied**. Policy cannot target heat alone — it must target heat × physiology.


In [ ]:

merged['face_industrial'] = merged['residuals'] > 2
merged['face_lonely'] = merged['onePersonHousehold'] > merged['onePersonHousehold'].quantile(0.8)
merged['face_elderly'] = (merged['oldFemale'] + merged['oldMale']) > (merged['oldFemale'] + merged['oldMale']).quantile(0.8)

# Calculate number of faces per tract
merged['num_faces'] = merged[['face_industrial', 'face_lonely', 'face_elderly']].sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, col, title, color in zip(axes, ['face_industrial', 'face_lonely', 'face_elderly'],
                                 ['🏭 Industrial Periphery', '🚪 Loneliness Trap', '👴 Elderly Paradox'],
                                 ['red', 'purple', 'orange']):
    merged[merged[col]].plot(ax=ax, color=color, markersize=5)
    ax.set_title(title, fontsize=12); ax.set_axis_off()

plt.suptitle('Three Faces of Climate Vulnerability in Milan', fontsize=16, fontweight='bold')
plt.show()

# Multi-risk population math
POP_PER_TRACT = 1_400_000 / 6079
one_face = (merged['num_faces'] == 1).sum()
multi_face = (merged['num_faces'] >= 2).sum()
total_risk = (merged['num_faces'] >= 1).sum()

print(f"1 face: {one_face} tracts (~{int(one_face * POP_PER_TRACT):,} people)")
print(f"2+ faces: {multi_face} tracts (~{int(multi_face * POP_PER_TRACT):,} people)")
print(f"Total in at least 1 risk zone: {total_risk} tracts (~{int(total_risk * POP_PER_TRACT):,} people)")



**The Multi-Risk Population:**
- 1 risk factor: **~511,728 people** (2222 tracts)
- 2+ overlapping risk factors: **~59,417 people** (258 tracts)
**Interpretation:** These three faces barely overlap spatially. A one-size-fits-all policy will fail. We need three distinct social and infrastructural interventions.

## 16. Model Validation
To prove our physical model isn't overfitted, we train it on 80% of the data and test it on the unseen 20%.


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

# Prepare data
X_val = merged[['social_vulnerability', 'impervious_x', 'tree_cover_density']].dropna()
y_val = merged.loc[X_val.index, 'mean_lst']

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(X_val, y_val, test_size=0.2, random_state=42)

# Train
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict
preds = lr.predict(X_test)

# Calculate MAPE
mape = mean_absolute_percentage_error(y_test, preds)
print(f"Model Validation (20% Hold-out):")
print(f"Mean Absolute Percentage Error (MAPE): {mape*100:.2f}%")
print("-> This confirms our physical model's predictions are highly accurate for urban heat mapping.")



**Result:** The Mean Absolute Percentage Error (MAPE) is only **2.01%**.
*(The naive random-split MAPE is 2.01%; spatial block cross-validation gives 2.07% and provides a more spatially honest estimate.)*

## 17. The Ultimate Actionable Output: Priority Zones
We combine Heat, Social Vulnerability, and Industrial Anomalies into a single Priority Score. Here are the Top 20 census tracts that require immediate intervention.




In [ ]:

# 1. Create a composite priority score (Higher = Needs urgent intervention)
# Normalize variables to 0-1 scale
merged['norm_lst'] = merged['mean_lst'].rank(pct=True)
merged['norm_svi'] = merged['social_vulnerability'].rank(pct=True)
merged['norm_anomaly'] = merged['residuals'].rank(pct=True)

# Weight them equally
merged['priority_score'] = (merged['norm_lst'] + merged['norm_svi'] + merged['norm_anomaly']) / 3

# 2. Find Top 20 tracts
top_20 = merged.nlargest(20, 'priority_score')

# 3. Plot
fig, ax = plt.subplots(figsize=(12, 10))
merged.plot(ax=ax, color='#e0e0e0', edgecolor='white', linewidth=0.2)
top_20.plot(ax=ax, color='#e74c3c', edgecolor='black', linewidth=1)

# Add annotations for the top 3
for idx, row in top_20.head(3).iterrows():
    ax.annotate(f"Tract {row['SEZ2011']}\n{row['mean_lst']:.1f}°C",
                xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                xytext=(10, 10), textcoords="offset points",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8),
                arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))

ax.set_title('Top 20 Priority Tracts for Immediate Climate Intervention', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.show()

print("\n🚨 Top 3 Urgent Targets for Milan Municipality:")
print(top_20[['SEZ2011', 'mean_lst', 'social_vulnerability', 'residuals']].head(3).to_string(index=False))



This map is the ultimate tool for the city. The top municipal target is **Tract 151460001206**. The municipality can deploy cooling centers and medical monitoring directly to priority tracts (including this tract, which reaches 53.4°C and has high vulnerability).

## 18. Interactive Policy Dashboard
An interactive dashboard for city planners to explore temperature, vulnerability, and intervention simulations.


In [ ]:

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
import json

merged_plot = merged.to_crs(epsg=4326).copy()
merged_plot['num_faces'] = merged_plot['num_faces'].fillna(0).astype(int)
merged_plot['social_vulnerability'] = merged_plot['social_vulnerability'].fillna(0)
merged_plot['mean_lst'] = merged_plot['mean_lst'].fillna(0)
geojson = json.loads(merged_plot.to_json())

html_kpi = """
<div style="display: flex; justify-content: space-around; background-color: #f8f9fa; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin-bottom: 20px; font-family: sans-serif;">
    <div style="text-align: center;">
        <h2 style="color: #e74c3c; margin: 0; font-size: 36px;">571,146</h2>
        <p style="margin: 0; color: #555; font-size: 14px;">People in Risk Zones (40.8%)</p>
    </div>
    <div style="text-align: center; border-left: 2px solid #ddd; border-right: 2px solid #ddd; padding: 0 40px;">
        <h2 style="color: #e67e22; margin: 0; font-size: 36px;">59.2%</h2>
        <p style="margin: 0; color: #555; font-size: 14px;">Heat Anomalies in Industrial Zones</p>
    </div>
    <div style="text-align: center;">
        <h2 style="color: #8e44ad; margin: 0; font-size: 36px;">+0.858°C</h2>
        <p style="margin: 0; color: #555; font-size: 14px;">Heating Rate (Poor, per decade)</p>
    </div>
</div>
"""
display(HTML(html_kpi))

display(HTML("<h3 style='font-family: sans-serif; color: #333; margin-top: 20px;'>🗺️ 1. Interactive Heat Map & Vulnerability</h3>"))
fig_map = px.choropleth_mapbox(
    merged_plot, geojson=geojson, locations='SEZ2011', featureidkey='properties.SEZ2011',
    color='mean_lst', color_continuous_scale='YlOrRd', range_color=(35, 55),
    mapbox_style="open-street-map",
    zoom=10.5, center={"lat": 45.4642, "lon": 9.1900}, opacity=0.75,
    labels={'mean_lst':'Temp (°C)', 'num_faces':'Risks', 'social_vulnerability':'SVI'},
    hover_data={'SEZ2011': True, 'mean_lst': ':.2f', 'social_vulnerability': ':.2f', 'num_faces': True},
    title='Surface Temperature & Overlapping Vulnerabilities'
)
fig_map.update_layout(margin={"r":0,"t":40,"l":0,"b":0}, height=600)
display(fig_map)

display(HTML("<h3 style='font-family: sans-serif; color: #333; margin-top: 20px;'>🛠️ 2. Policy Simulation: Trees vs Cool Roofs</h3>"))
fig_sim = go.Figure(data=[go.Bar(
    x=['Current Reality', 'Trees Only', 'Cool Roofs'],
    y=[current, trees_t, roofs_t],
    marker_color=['#e74c3c', '#f39c12', '#3498db'],
    text=[f'{current:.2f}°C', f'{trees_t:.2f}°C ({trees_t-current:+.2f})',
          f'{roofs_t:.2f}°C ({roofs_t-current:+.2f})'], textposition='auto'
)])
fig_sim.update_layout(
    title='Intervention Simulation in Poor Areas',
    yaxis_title='Mean Surface Temperature (°C)', height=500,
    yaxis=dict(range=[min(current, trees_t, roofs_t) - 1,
                      max(current, trees_t, roofs_t) + 1])
)
display(fig_sim)



## 19. Policy Recommendations

Our analysis reveals that Milan's climate crisis is not a monolith. It is a combination of physical infrastructure and social isolation. Here is our three-pillar strategy:

### 🏭 1. The Industrial West (Hidden Heat Sources)
- **Evidence:** 59.2% of extreme heat anomalies overlap with industrial zones (a 4.3x concentration above baseline).
- **Action:** Mandatory green buffer zones around factories. Retrofit industrial roofs. Enforce strict industrial heat emission regulations.

### 🧱 2. The Eco-Structural Lock (Stop Planting Trees)
- **Evidence:** 64.3% of poor tracts are physically maxed out on space (no room for trees). Simulating tree planting yields +0.04°C.
- **Action:** Redirect tree-planting budgets in dense areas to **Cool Roofs (-0.36°C effect)**, reflective pavements, and green walls.

### 🚪 3. The Loneliness Trap (Social Policy, Not Physical)
- **Evidence:** When controlling for physical factors, the direct effect of SVI vanishes. "One-person households" is the strongest predictor of heat (+0.083°C).
- **Action:** Launch a "Check Your Neighbor" heat-wave registry. Subsidize AC for isolated seniors. Expand public transport to cooling centers.

**By combining NASA Earth Observation with social data, we have transformed an abstract climate problem into a precision public health tool.**


In [ ]:

# Install any missing libraries
!pip install mgwr -q
!pip install esda libpysal -q
!pip install geopandas -q

import ee
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Initialize Earth Engine if it is not already initialized
try:
    ee.Initialize(project='gen-lang-client-0068284339')
    print("✅ GEE initialized")
except:
    ee.Authenticate()
    ee.Initialize(project='gen-lang-client-0068284339')

print("✅ Libraries loaded")

print("📥 Загружаем GHSL population (JRC) за 2020 год...")



## 20. GHSL Population by Census Tract

Adds 2020 GHSL population estimates for each tract and summarizes their distribution.


In [ ]:

# GHSL Population 2020, 100m resolution
ghsl_img = ee.Image('JRC/GHSL/P2023A/GHS_POP/2020')

# Retrieve population estimates for each tract in batches
def get_population_by_tracts(gdf, image, batch_size=2000):
    all_records = []
    for start in range(0, len(gdf), batch_size):
        end = min(start + batch_size, len(gdf))
        features = [
            ee.Feature(
                ee.Geometry(r['geometry'].__geo_interface__),
                {'SEZ2011': str(r['SEZ2011'])}
            )
            for _, r in gdf.iloc[start:end].iterrows()
        ]
        try:
            data = image.reduceRegions(
                collection=ee.FeatureCollection(features),
                reducer=ee.Reducer.sum(),    # Sum the population within each tract
                scale=100
            ).getInfo()
            for f in data['features']:
                all_records.append({
                    'SEZ2011': f['properties']['SEZ2011'],
                    'population_ghsl': f['properties'].get('sum')
                })
            print(f"  ✅ {end}/{len(gdf)}")
        except Exception as e:
            print(f"  ⚠️ Error: {e}")
    return pd.DataFrame(all_records)

pop_df = get_population_by_tracts(svi_wgs84, ghsl_img)
pop_df['SEZ2011'] = pop_df['SEZ2011'].astype(str).str.strip()

# Merge the population estimates into the tract data
merged = merged.merge(pop_df, on='SEZ2011', how='left')
merged['population_ghsl'] = merged['population_ghsl'].fillna(0).astype(int)

# Compare GHSL population with the previous estimate
total_pop = merged['population_ghsl'].sum()
print(f"\n📊 РЕАЛЬНОЕ НАСЕЛЕНИЕ (GHSL): {total_pop:,} человек")
print(f"📊 Старая оценка (uniform): {len(merged) * 230:,} человек")
print(f"📊 Отклонение: {abs(total_pop - len(merged) * 230) / total_pop * 100:.1f}%")

# Check population estimates for outliers
print(f"\nEDA проверка населения:")
print(f"  Min: {merged['population_ghsl'].min():,}")
print(f"  Max: {merged['population_ghsl'].max():,}")
print(f"  Median: {merged['population_ghsl'].median():,.0f}")
print(f"  Пустых: {(merged['population_ghsl']==0).sum()} трактов")



## 21. 24-Year MODIS LST and NDVI Trends

Compares summer LST and vegetation index across vulnerability quartiles from 2000 to 2023.


In [ ]:

print("📥 Загружаем MODIS LST (MOD11A2, 2000-2023)...")

# MOD11A2 = 8-day LST at 1km
modis = ee.ImageCollection('MODIS/061/MOD11A2') \
    .filterBounds(milan_bbox) \
    .filterDate('2000-01-01', '2023-12-31') \
    .select('LST_Day_1km')

# Return mean summer LST for a given year
years_modis = list(range(2000, 2024))

def get_modis_summer(year):
    start = f'{year}-06-01'
    end = f'{year}-08-31'
    coll = ee.ImageCollection('MODIS/061/MOD11A2') \
        .filterBounds(milan_bbox) \
        .filterDate(start, end) \
        .select('LST_Day_1km')
    # Convert raw MODIS LST values to degrees Celsius
    lst = coll.mean().multiply(0.02).subtract(273.15).rename('LST_Celsius')
    return lst.set('year', year)

print("⏳ Скачиваем 24 года MODIS (5-10 минут)...")

# Aggregate values for the Q1 and Q4 vulnerability groups
q1_fc_modis = gdf_to_ee_fc(svi_wgs84[svi_wgs84['svi_quartile'] == 'Q1 (Rich)'])
q4_fc_modis = gdf_to_ee_fc(svi_wgs84[svi_wgs84['svi_quartile'] == 'Q4 (Poor)'])

modis_q1, modis_q4 = [], []

for year in years_modis:
    print(f"  Обработка {year}...")
    img = get_modis_summer(year)
    try:
        d1 = img.reduceRegion(ee.Reducer.mean(), q1_fc_modis.geometry(), 1000, maxPixels=1e9).getInfo()
        d4 = img.reduceRegion(ee.Reducer.mean(), q4_fc_modis.geometry(), 1000, maxPixels=1e9).getInfo()
        modis_q1.append(d1.get('LST_Celsius', np.nan))
        modis_q4.append(d4.get('LST_Celsius', np.nan))
    except Exception as e:
        print(f"  ⚠️ {year}: {e}")
        modis_q1.append(np.nan)
        modis_q4.append(np.nan)

modis_ts = pd.DataFrame({'Year': years_modis, 'Q1_Rich': modis_q1, 'Q4_Poor': modis_q4})
modis_ts.to_csv("modis_time_series_2000_2023.csv", index=False)
print(f"\n✅ Сохранено: modis_time_series_2000_2023.csv")
print(modis_ts.head(10))

print("📥 Загружаем MODIS NDVI (MOD13A2, 2000-2023)...")

def get_modis_ndvi_summer(year):
    start = f'{year}-06-01'
    end = f'{year}-08-31'
    coll = ee.ImageCollection('MODIS/061/MOD13A2') \
        .filterBounds(milan_bbox) \
        .filterDate(start, end) \
        .select('NDVI')
    # Apply the scale factor to raw NDVI values
    ndvi = coll.mean().multiply(0.0001).rename('NDVI')
    return ndvi.set('year', year)

print("⏳ Скачиваем NDVI (5-10 минут)...")
ndvi_q1, ndvi_q4 = [], []

for year in years_modis:
    print(f"  {year}...")
    img = get_modis_ndvi_summer(year)
    try:
        d1 = img.reduceRegion(ee.Reducer.mean(), q1_fc_modis.geometry(), 1000, maxPixels=1e9).getInfo()
        d4 = img.reduceRegion(ee.Reducer.mean(), q4_fc_modis.geometry(), 1000, maxPixels=1e9).getInfo()
        ndvi_q1.append(d1.get('NDVI', np.nan))
        ndvi_q4.append(d4.get('NDVI', np.nan))
    except:
        ndvi_q1.append(np.nan); ndvi_q4.append(np.nan)

ndvi_ts = pd.DataFrame({'Year': years_modis, 'Q1_NDVI': ndvi_q1, 'Q4_NDVI': ndvi_q4})
ndvi_ts.to_csv("modis_ndvi_2000_2023.csv", index=False)
print(f"\n✅ Сохранено: modis_ndvi_2000_2023.csv")
print(ndvi_ts.head())

print("🔍 EDA: Проверка новых данных\n")

# 0. Check for and create the required columns
print("Доступные колонки:", sorted(merged.columns.tolist())[:30], "...\n")

# Recreate svi_quartile in case it was lost during a merge
if 'svi_quartile' not in merged.columns:
    if 'social_vulnerability' in merged.columns:
        merged['svi_quartile'] = pd.qcut(
            merged['social_vulnerability'], q=4,
            labels=['Q1 (Rich)', 'Q2', 'Q3', 'Q4 (Poor)']
        )
        print("✅ Создана svi_quartile")
    else:
        print("❌ social_vulnerability не найдена!")

# Recreate svi_q2 if it was lost during a merge
if 'svi_q2' not in merged.columns and 'social_vulnerability' in merged.columns:
    merged['svi_q2'] = pd.qcut(merged['social_vulnerability'], q=4,
                                labels=['Rich','Q2','Q3','Poor'])

# Create lst_q for the cross-tabulation
if 'lst_q' not in merged.columns and 'mean_lst' in merged.columns:
    merged['lst_q'] = pd.qcut(merged['mean_lst'], q=4, labels=['Cold','Mild','Warm','Hot'])

# Required columns
required = ['mean_lst','social_vulnerability','impervious_x',
            'tree_cover_density','population_ghsl','svi_quartile']
missing = [c for c in required if c not in merged.columns]
if missing:
    print(f"⚠️ КРИТИЧНО: отсутствуют {missing}")
    raise ValueError(f"Missing: {missing}")
print("✅ Все критические колонки на месте\n")

# 1. Population distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

pop_nonzero = merged[merged['population_ghsl'] > 0]
axes[0].hist(pop_nonzero['population_ghsl'], bins=50, color='#3498db', edgecolor='black')
axes[0].set_title('GHSL Population Distribution (nonzero only)')
axes[0].set_xlabel('People per tract')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

merged.boxplot(column='population_ghsl', by='svi_quartile', ax=axes[1])
axes[1].set_title('Population by SVI Quartile')
axes[1].set_ylabel('People per tract')
axes[1].set_xticklabels(['Q1\n(Rich)','Q2','Q3','Q4\n(Poor)'])
plt.suptitle('')
plt.tight_layout()
plt.show()

# 2. Correlations
print(f"Корреляция population × SVI: {merged['population_ghsl'].corr(merged['social_vulnerability']):.3f}")
print(f"Корреляция population × LST: {merged['population_ghsl'].corr(merged['mean_lst']):.3f}")

# 3. Tracts with zero population
empty_tracts = (merged['population_ghsl'] == 0).sum()
print(f"\n🚨 Трактов с населением 0: {empty_tracts} (это парки, промзоны, аэропорт — их не надо исключать из heat analysis, но НЕЛЬЗЯ учитывать в population-weighted)")

# 4. MODIS data checks
print(f"\nMODIS LST TS: {len(modis_ts)} лет, пропусков: {modis_ts['Q1_Rich'].isna().sum()}")
print(f"MODIS LST Q1 range: [{modis_ts['Q1_Rich'].min():.1f}, {modis_ts['Q1_Rich'].max():.1f}]°C")
print(f"MODIS LST Q4 range: [{modis_ts['Q4_Poor'].min():.1f}, {modis_ts['Q4_Poor'].max():.1f}]°C")
print(f"⚠️ MODIS LST ниже Landsat на ~8-10°C — это НОРМАЛЬНО: 1km пиксель усредняет, Landsat 100m видит hot spots.\n")

# 5. NDVI data checks
print(f"MODIS NDVI TS: {len(ndvi_ts)} лет, пропусков: {ndvi_ts['Q1_NDVI'].isna().sum()}")
print(f"NDVI Q1: {ndvi_ts['Q1_NDVI'].iloc[0]:.4f} → {ndvi_ts['Q1_NDVI'].iloc[-1]:.4f}")
print(f"NDVI Q4: {ndvi_ts['Q4_NDVI'].iloc[0]:.4f} → {ndvi_ts['Q4_NDVI'].iloc[-1]:.4f}")
print(f"NDVI gap (Q1-Q4): {(ndvi_ts['Q1_NDVI'] - ndvi_ts['Q4_NDVI']).mean():.4f} — стабилен ли?")

import pymannkendall as mk
from scipy.stats import theilslopes

print("📊 Анализ 24-летнего тренда MODIS\n")

# Remove missing values
modis_clean = modis_ts.dropna()

# Mann-Kendall
res_q1 = mk.original_test(modis_clean['Q1_Rich'])
res_q4 = mk.original_test(modis_clean['Q4_Poor'])

slope_q1, intercept_q1, lo_q1, hi_q1 = theilslopes(modis_clean['Q1_Rich'], modis_clean['Year'])
slope_q4, intercept_q4, lo_q4, hi_q4 = theilslopes(modis_clean['Q4_Poor'], modis_clean['Year'])

print("="*60)
print("SEN'S SLOPE (24 года MODIS)")
print("="*60)
print(f"Q1 (Rich): {slope_q1*10:+.3f} °C/decade (95% CI: {lo_q1*10:+.3f}, {hi_q1*10:+.3f})")
print(f"Q4 (Poor): {slope_q4*10:+.3f} °C/decade (95% CI: {lo_q4*10:+.3f}, {hi_q4*10:+.3f})")
print(f"\nMann-Kendall:")
print(f"  Q1: trend={res_q1.trend}, p={res_q1.p:.4f}")
print(f"  Q4: trend={res_q4.trend}, p={res_q4.p:.4f}")
print(f"\nGAP (Q4 - Q1) per decade: {(slope_q4 - slope_q1)*10:+.3f} °C")
print("="*60)

# Plot the results
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(modis_clean['Year'], modis_clean['Q1_Rich'], 'o-', color='#2ecc71',
        label=f'Q1 Rich: {slope_q1*10:+.3f}°C/dec', linewidth=2)
ax.plot(modis_clean['Year'], modis_clean['Q4_Poor'], 's-', color='#e74c3c',
        label=f'Q4 Poor: {slope_q4*10:+.3f}°C/dec', linewidth=2)

# Trend lines
yrs = modis_clean['Year'].values
ax.plot(yrs, intercept_q1 + slope_q1*yrs, '--', color='#2ecc71', alpha=0.5)
ax.plot(yrs, intercept_q4 + slope_q4*yrs, '--', color='#e74c3c', alpha=0.5)

ax.set_xlabel('Year')
ax.set_ylabel('Summer LST (°C) — MODIS')
ax.set_title('24-Year Trend: MODIS LST (2000-2023) — Rich vs Poor', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('modis_24yr_trend.png', dpi=150)
plt.show()



## 22. Geographically Weighted Regression

Estimates how the local relationship between vulnerability, impervious cover, tree cover, and temperature varies across Milan. The local SVI effect ranges from **−1.9°C to +1.8°C**.


In [ ]:

from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
import libpysal

print("🌍 GWR: Geographically Weighted Regression\n")

# Prepare the GWR input data
gwr_df = merged.dropna(subset=['mean_lst', 'social_vulnerability',
                                'impervious_x', 'tree_cover_density']).copy()
gwr_df = gwr_df.to_crs(epsg=32632)    # Use UTM zone 32N for distance-based coordinates

coords = np.array([(g.x, g.y) for g in gwr_df.geometry.centroid])
y = gwr_df['mean_lst'].values.reshape(-1, 1)
X = gwr_df[['social_vulnerability', 'impervious_x', 'tree_cover_density']].values

# Select the optimal bandwidth
print("⏳ Поиск оптимального bandwidth...")
bw = Sel_BW(coords, y, X).search(criterion='AICc')
print(f"✅ Bandwidth: {bw}")

# GWR
gwr_model = GWR(coords, y, X, bw)
gwr_results = gwr_model.fit()

# Extract local SVI coefficients (parameter index 1)
gwr_df['gwr_svi'] = gwr_results.params[:, 1]
gwr_df['gwr_impervious'] = gwr_results.params[:, 2]
gwr_df['gwr_tree'] = gwr_results.params[:, 3]

print(f"\nГлобальный OLS: SVI → LST = {gwr_results.params.mean(axis=0)[1]:.4f}")
print(f"GWR min/max: {gwr_df['gwr_svi'].min():.4f} / {gwr_df['gwr_svi'].max():.4f}")
print(f"GWR std: {gwr_df['gwr_svi'].std():.4f}")

# Visualize the results
fig, ax = plt.subplots(figsize=(10, 10))
gwr_df.plot(column='gwr_svi', cmap='RdBu_r', legend=True, ax=ax,
            legend_kwds={'label': 'Local SVI coefficient (°C)'})
ax.set_title('GWR: Where does vulnerability matter MOST?', fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('gwr_local_coef.png', dpi=150)
plt.show()

# Save the coefficient table
gwr_df[['SEZ2011', 'gwr_svi', 'gwr_impervious', 'gwr_tree']].to_csv('gwr_coefs.csv', index=False)



## 23. Spatial Block Cross-Validation

Evaluates model error using spatially grouped folds to reduce leakage between nearby tracts. Spatial block MAPE is **2.07%**, compared with the naive random-split MAPE of **2.01%**.


In [ ]:

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

print("🔄 Spatial Block Cross-Validation\n")

# Create a 5-by-5 grid of spatial blocks
gdf_cv = merged.dropna(subset=['mean_lst', 'social_vulnerability',
                                'impervious_x', 'tree_cover_density']).copy()

# Assign tracts to blocks using their coordinates
gdf_cv['block_x'] = pd.qcut(gdf_cv.geometry.centroid.x, q=5, labels=False)
gdf_cv['block_y'] = pd.qcut(gdf_cv.geometry.centroid.y, q=5, labels=False)
gdf_cv['block'] = gdf_cv['block_x'].astype(str) + '_' + gdf_cv['block_y'].astype(str)

X = gdf_cv[['social_vulnerability', 'impervious_x', 'tree_cover_density']].values
y = gdf_cv['mean_lst'].values
groups = gdf_cv['block'].values

gkf = GroupKFold(n_splits=5)
mapes = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    model = LinearRegression()
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    mape = mean_absolute_percentage_error(y[test_idx], preds) * 100
    mapes.append(mape)
    print(f"  Fold {fold+1}: MAPE = {mape:.2f}%")

print(f"\n📊 HONEST MAPE (spatial CV): {np.mean(mapes):.2f}% ± {np.std(mapes):.2f}%")
print(f"📊 Naive MAPE (random split, old): 2.01%")
print(f"📊 Реальное падение точности: ~{(np.mean(mapes) - 2.01):.2f} п.п.")



## 24. Bivariate LISA

Measures spatial association between heat and social vulnerability and maps local cluster categories. The video reports **589 Red tracts** (High Heat + High Vulnerability) and **687 Orange tracts** (Low Heat + High Vulnerability, the invisible majority).


In [ ]:

from esda.moran import Moran_BV, Moran_Local_BV
import libpysal

print("🔬 Bivariate LISA: Heat × Vulnerability\n")

# Build the spatial weights matrix
w = libpysal.weights.Queen.from_dataframe(merged)
w.transform = 'r'

# Bivariate Moran's I
moran_bv = Moran_BV(merged['mean_lst'], merged['social_vulnerability'], w)
print(f"Bivariate Moran's I: {moran_bv.I:.4f}")
print(f"p-value (perm): {moran_bv.p_sim:.4f}")
# Use the z-score because newer esda versions do not expose EI
print(f"z-score: {moran_bv.z_sim:.4f}")

# Local Bivariate LISA
lisa_bv = Moran_Local_BV(merged['mean_lst'], merged['social_vulnerability'], w)
merged['biv_q'] = lisa_bv.q
merged['biv_p'] = lisa_bv.p_sim

# Cluster labels: 1=HH, 2=LL, 3=LH, 4=HL
merged['biv_cat'] = np.where(merged['biv_p'] > 0.05, 'Not Sig.',
                     np.where(merged['biv_q'] == 1, 'High Heat + High Vuln',
                     np.where(merged['biv_q'] == 2, 'Low Heat + Low Vuln',
                     np.where(merged['biv_q'] == 3, 'Low Heat + High Vuln',
                              'High Heat + Low Vuln'))))

print(f"\nРаспределение по категориям:")
print(merged['biv_cat'].value_counts())

# Visualize the results
fig, ax = plt.subplots(figsize=(10, 10))
colors = {
    'High Heat + High Vuln': '#e74c3c',
    'Low Heat + Low Vuln': '#3498db',
    'Low Heat + High Vuln': '#f39c12',
    'High Heat + Low Vuln': '#9b59b6',
    'Not Sig.': '#e0e0e0'
}
for cat, col in colors.items():
    sub = merged[merged['biv_cat'] == cat]
    if not sub.empty:
        sub.plot(ax=ax, color=col, edgecolor='black', linewidth=0.2, label=cat)
ax.set_title(f'Bivariate LISA: Heat × Vulnerability (I={moran_bv.I:.3f})', fontweight='bold')
ax.legend(loc='lower right')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('bivariate_lisa.png', dpi=150)
plt.show()



## 25. Compound Climate Exposure

Combines ranked heat, vulnerability, NO2, inverse tree cover, and population into a tract-level score.


In [ ]:

print("🧮 Композитный индекс уязвимости\n")

# Normalize the key variables to percentile ranks
merged['norm_lst'] = merged['mean_lst'].rank(pct=True)
merged['norm_svi'] = merged['social_vulnerability'].rank(pct=True)
merged['norm_no2'] = merged['no2'].rank(pct=True) if 'no2' in merged.columns else 0.5
merged['norm_tree_inv'] = 1 - merged['tree_cover_density'].rank(pct=True)    # Invert the tree-cover percentile rank
merged['norm_pop'] = merged['population_ghsl'].rank(pct=True)

# Calculate the weighted compound exposure score
merged['compound_exposure'] = (
    0.30 * merged['norm_lst'] +
    0.25 * merged['norm_svi'] +
    0.20 * merged['norm_no2'] +
    0.15 * merged['norm_tree_inv'] +
    0.10 * merged['norm_pop']
)

# Select the top 20 tracts
top_comp = merged.nlargest(20, 'compound_exposure')
print(f"Топ-5 композитных трактов:")
print(top_comp[['SEZ2011', 'mean_lst', 'social_vulnerability',
                'population_ghsl', 'compound_exposure']].head(5).to_string(index=False))

# Calculate population-weighted exposure
# Count residents in the top decile of compound exposure
threshold = merged['compound_exposure'].quantile(0.9)
in_risk = merged[merged['compound_exposure'] >= threshold]
total_people_in_risk = in_risk['population_ghsl'].sum()
total_city = merged['population_ghsl'].sum()

print(f"\n🚨 POPULATION-WEIGHTED EXPOSURE:")
print(f"  Трактов в топ-10%: {len(in_risk)}")
print(f"  Людей в них: {total_people_in_risk:,}")
print(f"  Доля города: {total_people_in_risk/total_city*100:.1f}%")
print(f"  (Старая оценка была: {len(in_risk) * 230:,} человек — расхождение!)")

# Visualize the results
fig, ax = plt.subplots(figsize=(12, 10))
merged.plot(column='compound_exposure', cmap='inferno', legend=True, ax=ax,
            legend_kwds={'label': 'Compound Exposure Score'})
ax.set_title('Compound Climate Exposure Index (population-weighted)', fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('compound_exposure.png', dpi=150)
plt.show()

import statsmodels.formula.api as smf



## 26. Quantile Regression

Compares the SVI association across selected points of the temperature distribution. The video reports **Q50 = +0.034°C**, **Q90 = +0.079°C**, and a **10x** increase at Q95.


In [ ]:

print("🎯 Quantile Regression (для экстремальной жары)\n")

df_qr = merged.dropna(subset=['mean_lst', 'social_vulnerability',
                               'impervious_x', 'tree_cover_density']).copy()
df_qr = df_qr.rename(columns={
    'mean_lst': 'LST', 'social_vulnerability': 'SVI',
    'impervious_x': 'Impervious', 'tree_cover_density': 'Tree'
})

print("Сравнение OLS и Quantile regression:\n")
print(f"{'Quantile':<12} {'SVI coef':<12} {'p-value':<10} {'Impervious':<12}")
print("-" * 50)

results_summary = []
for q in [0.5, 0.75, 0.90, 0.95]:
    m = smf.quantreg('LST ~ SVI + Impervious + Tree', df_qr).fit(q=q)
    svi_coef = m.params['SVI']
    svi_p = m.pvalues['SVI']
    imp_coef = m.params['Impervious']
    print(f"{q:<12} {svi_coef:+.4f}     {svi_p:.4f}    {imp_coef:+.4f}")
    results_summary.append({'q': q, 'svi': svi_coef, 'imp': imp_coef})

print("\nИнтерпретация:")
print("- Если коэффициент SVI растёт к 95-му перцентилю — уязвимость важнее для экстремальной жары.")
print("- Если падает — уязвимость влияет только на средние температуры.")

# Plot the results
qs = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
svi_coefs = [smf.quantreg('LST ~ SVI + Impervious + Tree', df_qr).fit(q=q).params['SVI'] for q in qs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(qs, svi_coefs, 'o-', color='#e74c3c', linewidth=3, markersize=10)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Quantile of LST distribution')
ax.set_ylabel('SVI coefficient (°C per SD)')
ax.set_title('Effect of Vulnerability Across the Temperature Distribution', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('quantile_regression.png', dpi=150)
plt.show()



## 27. Exporting Analysis Data

Exports tract-level results, time series, and local GWR coefficients for the Streamlit application.


In [ ]:

print("💾 Экспорт всех обновлённых данных для Streamlit app\n")

# Ensure geometries use the WGS84 coordinate reference system
merged_wgs84 = merged.to_crs(epsg=4326)

# Export the main GeoJSON with all application features
merged_wgs84.to_file("app_data.geojson", driver="GeoJSON")
print("✅ app_data.geojson сохранен")

# Export a geometry-free CSV for faster pandas loading
merged.drop(columns=['geometry']).to_csv("app_data.csv", index=False)
print("✅ app_data.csv сохранен")

# Export the 24-year MODIS time series
modis_ts.to_csv("time_series.csv", index=False)
print("✅ time_series.csv сохранен (MODIS 2000-2023)")

# Export the NDVI time series
ndvi_ts.to_csv("ndvi_time_series.csv", index=False)
print("✅ ndvi_time_series.csv сохранен")

# Reproject the GWR results and export the local coefficients
gwr_df_wgs = gwr_df.to_crs(epsg=4326)
gwr_cols = ['SEZ2011', 'gwr_svi', 'gwr_impervious', 'gwr_tree']
gwr_export = gwr_df_wgs[gwr_cols].copy()
gwr_export.to_csv("gwr_coefs.csv", index=False)
print("✅ gwr_coefs.csv сохранен")

print("\n📊 Итоговая статистика для игры:")
print(f"   Население (GHSL): {merged['population_ghsl'].sum():,}")
print(f"   Людей в топ-10% compound exposure: {total_people_in_risk:,}")
print(f"   Honest MAPE (spatial CV): {np.mean(mapes):.2f}%")
print(f"   Трактов в High Heat + High Vuln: {len(merged[merged['biv_cat'] == 'High Heat + High Vuln'])}")